The following runs inside the BenchMAX environment.

# Behavioral cloning: Kuhn poker

Load the pinned Kuhn poker expert, collect demonstrations with the public `game.rollout` API, and fit one small neural policy per player by maximum likelihood of the expert's **sampled actions**. All experiment code lives in this notebook; no BC algorithm is added to the library.

Kuhn poker is small enough to evaluate exact exploitability before and after cloning. A low supervised loss does not guarantee a low exploitability: demonstrations cover the expert's state distribution, and a clone can visit different states.

## Setup

From this repository, run `uv sync --extra training --group notebook` and launch `uv run jupyter lab`. Select that environment as the notebook kernel.

In [1]:
import warnings

# ignore all warnings for a cleaner overview
warnings.filterwarnings("ignore")

In [2]:
import torch

from benchmax import load_game

# Separate seeds for demonstrations, held-out episodes, initialization, and minibatches.
TRAIN_SEED, VALID_SEED, MODEL_SEED, BATCH_SEED = 7, 8, 9, 10
TRAIN_EPISODES, VALID_EPISODES = 4096, 1024
EPOCHS, BATCH_SIZE = 100, 256

torch.manual_seed(MODEL_SEED)
torch.set_num_threads(1)
device = torch.device("cpu")

game = load_game("kuhn_poker")

print("Game:", game.name, "\nPlayers:", game.players)

Game: kuhn_poker 
Players: (0, 1)


## Collect demonstrations

With `collect_data=True`, one call returns episode returns plus NumPy arrays grouped by player: observations, sampled action IDs, legal-action masks, episode IDs, and step IDs. No trajectory-unpacking loop is needed. In a sequential game, a player's dataset contains only their own decisions. The collector batches policy inference across environments and stores the decision arrays in memory.

The seed controls environment chance and action sampling. Changing inference batch size preserves sampled decisions for a fixed policy independent of batch composition, but may change row order. BC does not require episode-sorted rows. Rollout also supports bounded trajectories in continuing games. Here `require_complete=True` ensures our poker demonstrations are complete.

`load_expert` currently reads the public `ALPIuzh/benchmax-preview` Hugging Face bucket. Authentication is not required to download these policies. Verified cached policies can be reused offline. Authentication/download failures are surfaced rather than silently substituting a different teacher.

In [3]:
import warnings

from benchmax import load_expert

expert = load_expert(game)

_, train_data = game.rollout(expert, episodes=TRAIN_EPISODES, seed=TRAIN_SEED, batch_size=64,
                             collect_data=True, require_complete=True)
_, valid_data = game.rollout(expert, episodes=VALID_EPISODES, seed=VALID_SEED, batch_size=64,
                             collect_data=True, require_complete=True)

print({player: len(data["actions"]) for player, data in train_data.items()})

{0: 5707, 1: 4096}


## Build one supervised dataset per player

Use independently sampled episodes for training and validation. Every row uses the player's own observation, the legal-action mask, and the sampled action's **column index** in `game.action_spaces[player]`. Action IDs need not be consecutive integers. The model never receives global state or the opponent's hidden cards.

In [4]:
import numpy as np


def as_tensors(data, player):
    # Map action IDs to classifier columns without assuming consecutive IDs.
    columns = (data["actions"][:, None] == np.asarray(game.action_spaces[player])).argmax(axis=1)
    return (torch.as_tensor(data["observations"], dtype=torch.float32, device=device).flatten(start_dim=1),
            torch.as_tensor(data["legal_action_mask"], dtype=torch.bool, device=device),
            torch.as_tensor(columns, dtype=torch.long, device=device))


train_tensors = {player: as_tensors(train_data[player], player) for player in game.players}
valid_tensors = {player: as_tensors(valid_data[player], player) for player in game.players}

## Train with masked cross-entropy

Illegal logits are masked before computing the loss, matching policy inference. We learn a distribution rather than taking an argmax: mixed strategies matter in poker. Held-out negative log-likelihood need not approach zero because the teacher is stochastic. These small CPU training settings are illustrative, not tuned BC benchmark results.

In [5]:
from torch import nn

models = {
    player: nn.Sequential(nn.Linear(int(np.prod(game.observation_spaces[player])), 64), nn.Tanh(),
                          nn.Linear(64, len(game.action_spaces[player]))).to(device)
    for player in game.players
}
optimizers = {player: torch.optim.Adam(model.parameters(), lr=3e-3) for player, model in models.items()}
batch_rng = torch.Generator(device="cpu").manual_seed(BATCH_SEED)


def masked_loss(model, data):
    observations, mask, labels = data
    logits = model(observations).masked_fill(~mask, -torch.inf)
    return nn.functional.cross_entropy(logits, labels)


history = []
for epoch in range(EPOCHS):
    for player, model in models.items():
        model.train()
        data = train_tensors[player]
        order = torch.randperm(len(data[2]), generator=batch_rng)
        for indices in order.split(BATCH_SIZE):
            loss = masked_loss(model, tuple(values[indices] for values in data))
            optimizers[player].zero_grad()
            loss.backward()
            optimizers[player].step()

    with torch.no_grad():
        for model in models.values():
            model.eval()
        train_nll = {p: masked_loss(models[p], train_tensors[p]).item() for p in game.players}
        valid_nll = {p: masked_loss(models[p], valid_tensors[p]).item() for p in game.players}
    history.append((train_nll, valid_nll))
    if epoch == 0 or (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d}: train NLL {train_nll}, held-out NLL {valid_nll}")

Epoch   1: train NLL {0: 0.5065933465957642, 1: 0.49873316287994385}, held-out NLL {0: 0.5015806555747986, 1: 0.49089115858078003}
Epoch  20: train NLL {0: 0.41834962368011475, 1: 0.269867867231369}, held-out NLL {0: 0.41162917017936707, 1: 0.2586353123188019}
Epoch  40: train NLL {0: 0.41665560007095337, 1: 0.2650708258152008}, held-out NLL {0: 0.41029319167137146, 1: 0.2527373433113098}
Epoch  60: train NLL {0: 0.4155431389808655, 1: 0.2646338641643524}, held-out NLL {0: 0.4080630838871002, 1: 0.25198230147361755}
Epoch  80: train NLL {0: 0.4149034023284912, 1: 0.26510801911354065}, held-out NLL {0: 0.40615126490592957, 1: 0.25479379296302795}
Epoch 100: train NLL {0: 0.4140504002571106, 1: 0.264729768037796}, held-out NLL {0: 0.40690141916275024, 1: 0.25139373540878296}


## Expose the clone as a BenchMAX policy and evaluate

The adapter returns probabilities in the requested action-space order and sets illegal probabilities to exactly zero. `game.evaluate` checks the entire strategy profile, including information states absent from demonstrations. Compare exact exploitability with the selected teacher and a uniform baseline; BC is not guaranteed to improve on or exactly recover its teacher.

In [6]:
from benchmax import PolicyProfile


@torch.no_grad()
def cloned_policy(player, observations, action_space, legal_action_mask):
    if tuple(action_space) != tuple(game.action_spaces[player]):
        raise ValueError("Unexpected action-space order")

    model = models[player].eval()
    values = torch.as_tensor(observations, dtype=torch.float32, device=device).flatten(start_dim=1)
    mask = torch.as_tensor(legal_action_mask, dtype=torch.bool, device=device)
    logits = model(values).masked_fill(~mask, -torch.inf)
    return logits.softmax(dim=-1).cpu().numpy()


clone = PolicyProfile(cloned_policy)
uniform = PolicyProfile(lambda player, observations, action_space, mask: mask / mask.sum(axis=1, keepdims=True))

metrics = {
    name: game.evaluate(policy)
    for name, policy in (("uniform", uniform), ("expert", expert), ("BC clone", clone))
}

for name, values in metrics.items():
    print(f"{name:8s}: exploitability={values['exploitability']:.6f}, NashConv={values['nash_conv']:.6f}")

uniform : exploitability=0.458333, NashConv=0.916667
expert  : exploitability=0.011123, NashConv=0.022245
BC clone: exploitability=0.009210, NashConv=0.018419


## Interpret the results

Held-out loss measures imitation; exploitability measures strategic weakness. More demonstrations can improve the fit, but faithfully copying an exploitable expert also copies its weaknesses. Compare the clone with the expert, not only with zero exploitability.

To explore data coverage, vary `TRAIN_EPISODES` and repeat with several demonstration and model seeds, keeping validation data fixed. Compare held-out loss and exploitability together: individual runs need not improve monotonically. If plotting bands across seeds, label them as variation across runs, not confidence intervals for the mean.

The clone is an ordinary callable policy and can also be passed to `game.rollout(clone, episodes=10, seed=11)`. Its PyTorch network remains notebook-owned; the library's portable codecs do not automatically serialize arbitrary callable networks.

To use a newly trained expert instead, replace the `load_expert(game)` call with your trained `PolicyProfile` (for example, returned by `benchmax.training.train`). Training a strong teacher requires convergence checks and tuning; this notebook avoids retraining the pinned teacher. For larger games, scale demonstration coverage and validation deliberately, and use an appropriate evaluator rather than assuming exact traversal is inexpensive.